# 文本音频生成器 - Google Colab 应用程序

该应用程序通过人工智能旁白将书籍/文本转换为音频
并提供问答互动聊天界面。

使用的型号（针对 T4 GPU 优化 - ~16GB VRAM）：

- 法学硕士：Qwen/Qwen2.5-1.5B-指令（4 位）
- TTS：微软/speecht5_tts
- ASR：openai/whisper-tiny

作者：丹尼斯·恩古吉·加通杜

# 安装依赖项

# ===================================================================

In [ ]:
# 首先运行此单元以安装所有必需的软件包

# 升级点
!pip install -q --upgrade pip

# 核心机器学习堆栈
!pip install -q transformers accelerate bitsandbytes

# 语音T5 TTS
!pip install -q datasets

# 音频处理
!pip install -q librosa soundfile

# 文件处理
!pip install -q PyPDF2 python-docx sentencepiece protobuf

# UI
!pip install -q gradio>=4.0.0

In [ ]:
import torch

print("CUDA Available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# 导入和配置

# ===================================================================

In [ ]:
import os
import gc
import re
import json
import tempfile
import warnings
from dataclasses import dataclass, field
from enum import Enum, auto
from typing import Optional, List, Dict, Any, Tuple, Generator
from pathlib import Path

import torch
import numpy as np
import soundfile as sf
from google.colab import userdata

# 抱脸
from huggingface_hub import login

# 变形金刚和机器学习
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSpeechSeq2Seq,
    AutoProcessor,
    BitsAndBytesConfig,
    SpeechT5Processor,
    SpeechT5ForTextToSpeech,
    SpeechT5HifiGan,
)

# 音频处理
import librosa

# 降低高频刺耳声（更平静的音调）
import scipy.signal as signal

# UI 渐变
import gradio as gr

# 文件处理
import PyPDF2
from docx import Document

warnings.filterwarnings("ignore")


# 登录拥抱脸
hf_token = userdata.get("HF_TOKEN")
login(token=hf_token, add_to_git_credential=True)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# 配置

# ===================================================================

In [ ]:
@dataclass
class AppConfig:
    """具有模型名称和设置的应用程序配置。"""

    # 型号名称（T4 GPU 的所有非门控小型型号）
    LLM_MODEL: str = "Qwen/Qwen2.5-1.5B-Instruct"
    TTS_MODEL: str = "microsoft/speecht5_tts"
    ASR_MODEL: str = "openai/whisper-tiny"

    # 生成设置
    MAX_TOKENS: int = 2048
    TEMPERATURE: float = 0.7
    TOP_P: float = 0.9

    AUDIO_SAMPLE_RATE: int = 16000

    # 部分设置
    MAX_SECTION_LENGTH: int = 600  # Characters per section
    MIN_SECTION_LENGTH: int = 200  # Minimum for a valid section

# 模型经理

#====================================================================

In [ ]:
class ModelState(Enum):
    """模型的可能状态。"""

    UNLOADED = auto()
    LOADED = auto()
    LOADING = auto()
    ERROR = auto()


@dataclass
class ModelContainer:
    """已加载模型及其组件的容器。"""

    name: str
    model: Optional[Any] = None
    tokenizer: Optional[Any] = None
    processor: Optional[Any] = None
    state: ModelState = ModelState.UNLOADED
    error_message: Optional[str] = None


class ModelManager:
    """T4 GPU 的内存高效模型管理器。

    管理模型的加载/卸载以保持在 VRAM 限制内。
    为音频生成与聊天提供基于阶段的加载。"""

    def __init__(self, config: AppConfig):
        self.config = config
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        # 初始化模型容器
        self._models: Dict[str, ModelContainer] = {
            "llm": ModelContainer(name=self.config.LLM_MODEL),
            "tts": ModelContainer(name=self.config.TTS_MODEL),
            "asr": ModelContainer(name=self.config.ASR_MODEL),
        }

        # LLM 的量化配置
        self._quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4",
        )

        print(f"🖥️ Device: {self.device}")
        if torch.cuda.is_available():
            print(f"📊 GPU: {torch.cuda.get_device_name(0)}")
            print(
                f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB"
            )

    def _clear_memory(self):
        """强制垃圾收集和 GPU 缓存清除。"""
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
            torch.cuda.synchronize()

    def get_memory_usage(self) -> Dict[str, float]:
        """获取当前 GPU 内存使用情况（以 GB 为单位）。"""
        if not torch.cuda.is_available():
            return {"allocated": 0, "reserved": 0, "free": 0}

        allocated = torch.cuda.memory_allocated(0) / 1e9
        reserved = torch.cuda.memory_reserved(0) / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9

        return {
            "allocated": round(allocated, 2),
            "reserved": round(reserved, 2),
            "free": round(total - reserved, 2),
        }

    # -------------------------------------------------------------------------
    # 法学硕士方法
    # -------------------------------------------------------------------------

    def load_llm(self) -> bool:
        """加载具有 4 位量化的 LLM。"""
        container = self._models["llm"]

        if container.state == ModelState.LOADED:
            return True

        try:
            container.state = ModelState.LOADING
            print(f"📥 Loading LLM ({container.name})...")

            container.tokenizer = AutoTokenizer.from_pretrained(
                container.name,
                trust_remote_code=True,
            )
            container.tokenizer.pad_token = container.tokenizer.eos_token

            container.model = AutoModelForCausalLM.from_pretrained(
                container.name,
                quantization_config=self._quant_config,
                device_map="auto",
                trust_remote_code=True,
            )

            container.state = ModelState.LOADED
            print(f"✅ LLM loaded! Memory: {self.get_memory_usage()}")
            return True

        except Exception as e:
            container.state = ModelState.ERROR
            container.error_message = str(e)
            print(f"❌ Failed to load LLM: {e}")
            return False

    def unload_llm(self):
        """卸载 LLM 并释放内存。"""
        container = self._models["llm"]
        if container.model is not None:
            del container.model
            container.model = None
        if container.tokenizer is not None:
            del container.tokenizer
            container.tokenizer = None
        container.state = ModelState.UNLOADED
        self._clear_memory()
        print("📤 LLM unloaded")

    # -------------------------------------------------------------------------
    # TTS 方法
    # -------------------------------------------------------------------------

    def load_tts(self) -> bool:
        """无需 SpeechBrain 即可加载 SpeechT5（稳定且干净）。"""

        container = self._models["tts"]

        if container.state == ModelState.LOADED:
            return True

        try:
            container.state = ModelState.LOADING
            print("📥 Loading SpeechT5...")

            container.processor = SpeechT5Processor.from_pretrained(container.name)

            container.model = SpeechT5ForTextToSpeech.from_pretrained(
                container.name
            ).to(self.device)

            container.vocoder = SpeechT5HifiGan.from_pretrained(
                "microsoft/speecht5_hifigan"
            ).to(self.device)

            # 🔥 使用固定稳定的扬声器嵌入（非零）
            torch.manual_seed(42)
            container.speaker_embeddings = torch.randn((1, 512)).to(self.device)

            container.state = ModelState.LOADED
            print("✅ SpeechT5 loaded successfully!")
            return True

        except Exception as e:
            container.state = ModelState.ERROR
            container.error_message = str(e)
            print(f"❌ Failed to load SpeechT5: {e}")
            return False

    def unload_tts(self):
        container = self._models["tts"]

        if hasattr(container, "model"):
            del container.model
        if hasattr(container, "processor"):
            del container.processor
        if hasattr(container, "vocoder"):
            del container.vocoder
        if hasattr(container, "speaker_embeddings"):
            del container.speaker_embeddings

        container.state = ModelState.UNLOADED
        self._clear_memory()
        print("📤 SpeechT5 unloaded")

    # -------------------------------------------------------------------------
    # 自动语音识别方法
    # -------------------------------------------------------------------------

    def load_asr(self) -> bool:
        """将 Whisper ASR 安全地加载到 float32（稳定）中。"""
        container = self._models["asr"]

        if container.state == ModelState.LOADED:
            return True

        try:
            container.state = ModelState.LOADING
            print(f"📥 Loading ASR ({container.name})...")

            container.processor = AutoProcessor.from_pretrained(container.name)

            container.model = AutoModelForSpeechSeq2Seq.from_pretrained(
                container.name,
                use_safetensors=True,
            ).to(self.device)

            container.state = ModelState.LOADED
            print("✅ ASR loaded successfully (float32)")
            return True

        except Exception as e:
            container.state = ModelState.ERROR
            container.error_message = str(e)
            print(f"❌ Failed to load ASR: {e}")
            return False

    def unload_asr(self):
        """卸载 ASR 模型并释放内存。"""
        container = self._models["asr"]
        if container.model is not None:
            del container.model
            container.model = None
        container.state = ModelState.UNLOADED
        self._clear_memory()
        print("📤 ASR unloaded")

    # -------------------------------------------------------------------------
    # 基于相位的加载
    # -------------------------------------------------------------------------

    def load_for_audio_generation(self) -> bool:
        """加载音频生成阶段所需的所有模型。
        模型：LLM、TTS"""
        print("\n" + "=" * 50)
        print("🔊 Loading Audio Generation Phase Models")
        print("=" * 50)

        self.unload_asr()  # Not needed for audio generation

        success = True
        success &= self.load_llm()
        success &= self.load_tts()

        if success:
            print("\n✅ All audio generation models loaded!")
            print(f"📊 Final memory: {self.get_memory_usage()}")
        else:
            print("\n❌ Some models failed to load")

        return success

    def load_for_chat(self) -> bool:
        """加载聊天阶段所需的模型。
        模型：LLM（保留）、ASR（添加）
        卸载：TTS"""
        print("\n" + "=" * 50)
        print("💬 Loading Chat Phase Models")
        print("=" * 50)

        # 卸载聊天不需要的重型模型
        self.unload_tts()

        success = True
        success &= self.load_llm()  # Keep loaded
        success &= self.load_asr()  # Add for voice input

        if success:
            print("\n✅ All chat models loaded!")
            print(f"📊 Final memory: {self.get_memory_usage()}")
        else:
            print("\n❌ Some models failed to load")

        return success

    def unload_all(self):
        """卸载所有模型。"""
        self.unload_llm()
        self.unload_tts()
        self.unload_asr()
        print("🧹 All models unloaded!")

    # -------------------------------------------------------------------------
    # 生成方法
    # -------------------------------------------------------------------------

    def generate_text(
        self,
        prompt: str,
        system_prompt: Optional[str] = None,
        max_new_tokens: Optional[int] = None,
    ) -> str:
        """使用法学硕士生成文本。"""
        container = self._models["llm"]

        if container.state != ModelState.LOADED:
            raise RuntimeError("LLM not loaded")

        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": prompt})

        print("  >> LLM: Applying chat template...")
        text = container.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        inputs = container.tokenizer(text, return_tensors="pt").to(
            container.model.device
        )

        generate_kwargs = {
            "max_new_tokens": max_new_tokens
            if max_new_tokens is not None
            else self.config.MAX_TOKENS,
            "temperature": self.config.TEMPERATURE,
            "top_p": self.config.TOP_P,
            "do_sample": True,
            "pad_token_id": container.tokenizer.eos_token_id,
        }

        print("  >> LLM: Generating response...")
        with torch.no_grad():
            outputs = container.model.generate(**inputs, **generate_kwargs)
        print("  >> LLM: Decoding response...")
        response = container.tokenizer.decode(
            outputs[0][inputs.input_ids.shape[1] :], skip_special_tokens=True
        )
        print("  >> LLM: Response generated.")

        return response.strip()

    def chunk_text_for_tts(self, text, max_tokens=600):
        sentences = re.split(r"(?<=[.!?])\s+", text)
        container = self._models["tts"]
        chunks = []
        current_chunk = []
        current_length = 0
        for sentence in sentences:
            sentence_tokens = container.processor.tokenizer(sentence)["input_ids"]
            sentence_length = len(sentence_tokens)
            if current_length + sentence_length > max_tokens:
                chunks.append(" ".join(current_chunk))
                current_chunk = [sentence]
                current_length = sentence_length
            else:
                current_chunk.append(sentence)
                current_length += sentence_length
        if current_chunk:
            chunks.append(" ".join(current_chunk))
        return chunks

    def generate_audio(
        self, text: str, description: str = None
    ) -> Tuple[np.ndarray, int]:
        """
        Generate natural, podcast-style TTS using SpeechT5 with gentle EQ and compression.
        """
        container = self._models["tts"]

        if container.state != ModelState.LOADED:
            raise RuntimeError("TTS model not loaded")

        text = re.sub(r"\s+", " ", text).strip()
        sentences = self.chunk_text_for_tts(text)

        audio_chunks = []

        for sentence in sentences:
            sentence = sentence.strip()
            if not sentence.split():
                continue
            # 标点符号标准化
            sentence = re.sub(r"[!?]{2,}", "!", sentence)
            if not sentence.endswith((".", "!", "?")):
                sentence += "."

            inputs = container.processor(text=sentence, return_tensors="pt").to(
                self.device
            )
            with torch.no_grad():
                speech = container.model.generate_speech(
                    inputs["input_ids"],
                    container.speaker_embeddings,
                    vocoder=container.vocoder,
                )

            audio_chunks.append(speech.cpu().numpy())

            # 在块之间插入短暂的停顿
            silence = np.zeros(int(0.55 * 16000))
            audio_chunks.append(silence)

        # 连接所有块
        audio = np.concatenate(audio_chunks).astype(np.float32)

        # =========================
        # 播客语气塑造
        # =========================

        fs = 16000
        nyquist = fs / 2

        high_cut = min(8000, nyquist * 0.99)
        b, a = signal.butter(2, high_cut / nyquist, btype="low")
        audio = signal.filtfilt(b, a, audio)

        # 2️⃣ 软动态范围压缩
        threshold = 0.6
        ratio = 2.0
        over_threshold = np.abs(audio) > threshold
        audio[over_threshold] = np.sign(audio[over_threshold]) * (
            threshold + (np.abs(audio[over_threshold]) - threshold) / ratio
        )

        # 3️⃣ 轻微增益提升，留出余量
        peak = np.max(np.abs(audio))
        if peak > 0:
            audio = audio / peak * 0.95  # leave some headroom

        return audio.astype(np.float32), fs

    def transcribe_audio(self, audio_path: str) -> str:
        """稳定的 Whisper 转录（无波形修改）。"""

        container = self._models["asr"]

        if container.state != ModelState.LOADED:
            raise RuntimeError("ASR model not loaded")

        # 加载干净的音频（Whisper 期望 16kHz 单声道）
        audio, _ = librosa.load(audio_path, sr=16000)

        inputs = container.processor(audio, sampling_rate=16000, return_tensors="pt")

        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            generated_ids = container.model.generate(**inputs, language="en")

        transcription = container.processor.batch_decode(
            generated_ids, skip_special_tokens=True
        )[0]

        return transcription

    # -------------------------------------------------------------------------
    # 易于访问的属性
    # -------------------------------------------------------------------------

    @property
    def llm_loaded(self) -> bool:
        return self._models["llm"].state == ModelState.LOADED

    @property
    def tts_loaded(self) -> bool:
        return self._models["tts"].state == ModelState.LOADED

    @property
    def asr_loaded(self) -> bool:
        return self._models["asr"].state == ModelState.LOADED

# 内容处理器

#====================================================================

In [ ]:
class ContentProcessor:
    """处理上传的文档并为有声读物生成做好准备。
    处理 PDF、DOCX、TXT 文件。"""

    def __init__(self, model_manager: ModelManager):
        self.manager = model_manager

    def extract_text(self, file_path: str) -> str:
        """从各种文件格式中提取文本。"""
        path = Path(file_path)
        suffix = path.suffix.lower()

        if suffix == ".pdf":
            return self._extract_from_pdf(file_path)
        elif suffix == ".docx":
            return self._extract_from_docx(file_path)
        elif suffix == ".txt":
            return self._extract_from_txt(file_path)
        else:
            raise ValueError(f"Unsupported file format: {suffix}")

    def _extract_from_pdf(self, file_path: str) -> str:
        """从 PDF 文件中提取文本。"""
        text = []
        with open(file_path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                text.append(page.extract_text())
        return "\n\n".join(text)

    def _extract_from_docx(self, file_path: str) -> str:
        """从 DOCX 文件中提取文本。"""
        doc = Document(file_path)
        return "\n\n".join([para.text for para in doc.paragraphs if para.text.strip()])

    def _extract_from_txt(self, file_path: str) -> str:
        """从 TXT 文件中提取文本。"""
        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()

    def create_sections(self, text: str, max_length: int = 600) -> List[Dict[str, str]]:
        """将文本分成逻辑部分以生成有声读物。

        返回带有“标题”和“内容”键的字典列表。"""
        system_prompt = """You are an expert educator who creates clear, engaging course content.
Given text from educational material, your task is to:
1. Break it into logical sections (each section should be a coherent topic)
2. Create a clear, concise title for each section
3. Ensure each section is educational and self-contained
4. Make sure that we have at least 10 sections so that they are not that long.

Format your response as JSON array:
[
  {"title": "Section Title", "content": "Section content here..."},
  ...
]"""

        prompt = f"""Please break the following educational content into logical sections.
Each section should be roughly {max_length} characters or less and cover a single coherent topic.

Content to process:
{text[:8000]}  # Limit to avoid token limits

Return ONLY the JSON array, no other text."""

        response = self.manager.generate_text(prompt, system_prompt)

        # 从响应中解析 JSON
        try:
            # 查找响应中的 JSON 数组
            json_match = re.search(r"\[.*\]", response, re.DOTALL)
            if json_match:
                sections = json.loads(json_match.group())
                return sections
        except json.JSONDecodeError:
            pass

        # 后备：简单的基于段落的分割
        return self._fallback_sections(text, max_length)

    def _fallback_sections(self, text: str, max_length: int) -> List[Dict[str, str]]:
        """如果 LLM 解析失败，则创建后备部分。"""
        paragraphs = text.split("\n\n")
        sections = []

        current_content = ""
        section_num = 1

        for para in paragraphs:
            para = para.strip()
            if not para:
                continue

            if len(current_content) + len(para) > max_length:
                if current_content:
                    sections.append(
                        {
                            "title": f"Section {section_num}",
                            "content": current_content.strip(),
                        }
                    )
                    section_num += 1
                    current_content = para
                else:
                    sections.append(
                        {"title": f"Section {section_num}", "content": para}
                    )
                    section_num += 1
            else:
                current_content += "\n\n" + para

        if current_content.strip():
            sections.append(
                {"title": f"Section {section_num}", "content": current_content.strip()}
            )

        return sections

    def generate_script(self, section: Dict[str, str]) -> str:
        """从一个部分生成一个旁白脚本。"""
        system_prompt = """You are an expert educator and scriptwriter.
Create clear, engaging narration scripts for educational audiobooks.
The script should:
- Be conversational and easy to understand
- Include clear explanations of concepts
- Use analogies and examples where helpful
- Be suitable for text-to-speech narration
- Be concise but comprehensive"""

        prompt = f"""Create a narration script for the following section.
The script should be suitable for an educational audiobook.

Section Title: {section["title"]}
Section Content: {section["content"]}

Write a clear, engaging narration script:"""

        return self.manager.generate_text(prompt, system_prompt)

# 音频发生器

# ===================================================================

In [ ]:
class AudioGenerator:
    """从处理的内容生成音频。"""

    def __init__(
        self, model_manager: ModelManager, content_processor: ContentProcessor
    ):
        self.manager = model_manager
        self.processor = content_processor

    def generate_section_audio(
        self, section: Dict[str, str], script: str, output_dir: str, section_num: int
    ) -> Dict[str, str]:
        """
        Generate audio components for a single section.

        Returns dict with paths to generated files.
        """
        print(f"\n🔊 Generating audio for section {section_num}: {section['title']}")

        paths = {}

        # 生成音频
        print("  🔊 Generating audio...")
        audio_data, sample_rate = self.manager.generate_audio(script)

        audio_path = os.path.join(output_dir, f"section_{section_num}_audio.wav")
        sf.write(audio_path, audio_data, sample_rate)
        paths["audio"] = audio_path

        # 计算音频持续时间
        audio_duration = len(audio_data) / sample_rate
        paths["duration"] = audio_duration

        print(f"  ✅ Section {section_num} complete! Duration: {audio_duration:.1f}s")

        return paths

    def compile_audiobook(
        self, section_files: List[Dict[str, str]], output_path: str
    ) -> str:
        """
        Compile all sections into a final audiobook (single audio file).
        """
        print("\n🎧 Compiling final audiobook...")

        all_audio_data = []
        sample_rate = self.manager.config.AUDIO_SAMPLE_RATE

        for i, files in enumerate(section_files, 1):
            print(f"  📌 Processing section {i}...")
            audio_file_path = files["audio"]
            audio, _ = librosa.load(audio_file_path, sr=sample_rate)
            all_audio_data.append(audio)

        # 连接所有音频数据
        print("  🔗 Concatenating audio clips...")
        concatenated_audio = np.concatenate(all_audio_data)

        # 写输出
        print(f"  💾 Saving audiobook to {output_path}...")
        sf.write(output_path, concatenated_audio, sample_rate)

        print(f"  ✅ Audiobook saved: {output_path}")

        return output_path

# 聊天管理器

# ===================================================================

In [ ]:
class ChatManager:
    """管理交互式聊天界面。
    保持与课程材料的对话上下文。"""

    def __init__(self, model_manager: ModelManager):
        self.manager = model_manager
        self.conversation_history: List[Dict[str, str]] = []
        self.course_context: str = ""
        self.sections: List[Dict[str, str]] = []

    def set_context(self, text: str, sections: List[Dict[str, str]]):
        """设置聊天的课程材料上下文。"""
        self.course_context = text[:10000]  # Limit context size
        self.sections = sections
        self.conversation_history = []  # Reset history for new material

    def ask(
        self, question: str, use_voice: bool = False, audio_path: Optional[str] = None
    ) -> Generator[str, None, None]:
        """
        Ask a question about the course material.

        Yields response chunks for streaming.
        """
        # 如果有语音输入，则转录音频
        if use_voice and audio_path:
            print("🎤 Transcribing voice input...")
            question = self.manager.transcribe_audio(audio_path)
            print(f"  📝 Transcribed: {question}")

        # 构建带有上下文的系统提示
        system_prompt = f"""You are a helpful educational assistant. You have access to course material
and should help the user understand it better.

Course Material Summary:
{self.course_context[:3000]}

Your role is to:
1. Answer questions about the material clearly and thoroughly
2. Provide additional explanations and examples when helpful
3. Relate answers back to the course content when relevant
4. Be encouraging and supportive in your teaching style"""

        # 添加对话历史记录
        messages = [{"role": "system", "content": system_prompt}]
        messages.extend(self.conversation_history)
        messages.append({"role": "user", "content": question})

        # 商店用户问题
        self.conversation_history.append({"role": "user", "content": question})

        # 生成响应
        response = self.manager.generate_text(question, system_prompt)

        # 店员回应
        self.conversation_history.append({"role": "assistant", "content": response})

        # 产量响应（可以针对实际流式传输进行修改）
        yield response

    def clear_history(self):
        """清除对话历史记录。"""
        self.conversation_history = []
        print("🧹 Conversation history cleared")



# 主要应用

# ===================================================================

In [ ]:
class CourseAudioApp:
    """将所有内容联系在一起的主要应用程序类。"""

    def __init__(self):
        print("🚀 Initializing Course Audio Generator...")

        self.config = AppConfig()
        self.manager = ModelManager(self.config)
        self.processor = ContentProcessor(self.manager)
        self.audio_gen = AudioGenerator(self.manager, self.processor)

        self.chat = ChatManager(self.manager)

        self.current_text: str = ""
        self.current_sections: List[Dict[str, str]] = []
        self.current_audio_path: Optional[str] = None

        print("✅ Application initialized!")

    def process_document(self, file_path: str) -> Tuple[str, str]:
        """处理上传的文档并返回摘要。"""
        print(f"\n📄 Processing document: {file_path}")

        # 确保加载 LLM 以生成摘要
        if not self.manager.llm_loaded:
            self.manager.load_llm()

        # 提取文本
        text = self.processor.extract_text(file_path)
        self.current_text = text

        # 生成摘要
        summary = self.manager.generate_text(
            f"Summarize the following educational content in 2-3 sentences:\n\n{text[:2000]}"
        )

        print(f"✅ Document processed! Length: {len(text)} characters")

        return f"Document loaded successfully!\n\nSummary: {summary}", text[:1000]

    def create_sections(self, progress=gr.Progress()) -> Tuple[str, str]:
        """从加载的文档创建部分。"""
        if not self.current_text:
            return "Please upload a document first!", ""

        progress(0.1, desc="Creating sections...")

        # 创建部分
        self.current_sections = self.processor.create_sections(self.current_text)

        progress(0.5, desc="Generating section summaries...")

        # 显示格式
        sections_text = "📚 Created Sections:\n\n"
        for i, section in enumerate(self.current_sections, 1):
            sections_text += f"**Section {i}: {section['title']}**\n"
            sections_text += f"{section['content'][:200]}...\n\n"

        progress(1.0, desc="Done!")

        return f"Created {len(self.current_sections)} sections!", sections_text

    def generate_audiobook(self, progress=gr.Progress()) -> str:
        """从各个部分生成完整的有声读物。"""
        if not self.current_sections:
            return "Please create sections first!"

        # 加载音频生成模型
        progress(0.0, desc="Loading models...")
        self.manager.load_for_audio_generation()

        # 为输出创建临时目录
        output_dir = tempfile.mkdtemp()
        section_files = []

        try:
            total_sections = len(self.current_sections)

            for i, section in enumerate(self.current_sections):
                progress(
                    (i + 0.5) / total_sections,
                    desc=f"Processing section {i + 1}/{total_sections}...",
                )

                # 生成脚本
                print(f"\n📝 Generating script for section {i + 1}...")
                script = self.processor.generate_script(section)

                # 生成音频组件
                progress(
                    (i + 0.8) / total_sections,
                    desc=f"Generating audio for section {i + 1}/{total_sections}...",
                )

                files = self.audio_gen.generate_section_audio(
                    section,
                    script,
                    output_dir,
                    i + 1,
                )
                section_files.append(files)

            # 编译最终有声读物
            progress(0.95, desc="Compiling final audiobook...")

            self.current_audio_path = os.path.join(output_dir, "course_audiobook.wav")
            final_audio_path = self.audio_gen.compile_audiobook(
                section_files, self.current_audio_path
            )

            progress(1.0, desc="Complete!")

            return final_audio_path

        except Exception as e:
            return f"Error generating audiobook: {str(e)}"

    def switch_to_chat(self) -> str:
        """切换到聊天模式（卸载重模型）。"""
        self.manager.load_for_chat()

        # 设置聊天上下文
        if self.current_text and self.current_sections:
            self.chat.set_context(self.current_text, self.current_sections)

        return "✅ Switched to chat mode! You can now ask questions about the material."

    def chat_response(self, message: str, history: List) -> str:
        """生成聊天响应。"""
        if not self.manager.llm_loaded:
            return "Please switch to chat mode first!"

        response = ""
        for chunk in self.chat.ask(message):
            response = chunk

        return response

    def voice_chat_response(self, audio_path: str, history: List) -> str:
        """从语音输入生成聊天响应。"""
        if not self.manager.asr_loaded:
            return "Please switch to chat mode first!"

        response = ""
        for chunk in self.chat.ask("", use_voice=True, audio_path=audio_path):
            response = chunk

        return response

# 渐变用户界面

# ===================================================================

In [ ]:
def create_ui():
    """创建渐变 UI。"""

    app = CourseAudioApp()

    with gr.Blocks(
        title="Course Audio Generator",
        theme=gr.themes.Soft(),
        css="""
        .header {text-align: center; margin-bottom: 20px;}
        .status {padding: 10px; border-radius: 5px; margin: 10px 0;}
        """,
    ) as demo:
        gr.Markdown(
            """
            # 🎓 课程音频生成器
            Transform your educational materials into engaging audio courses with AI narration.
            """
        )

        with gr.Tabs():
            # =================================================================
            # TAB 1：音频发生器
            # =================================================================
            with gr.TabItem("🎧 Audio Generator"):
                with gr.Row():
                    with gr.Column(scale=1):
                        gr.Markdown("### 📄 Upload Material")

                        file_input = gr.File(
                            label="Upload Document",
                            file_types=[".pdf", ".docx", ".txt"],
                        )

                        upload_btn = gr.Button("📤 Upload & Process", variant="primary")

                        upload_status = gr.Textbox(
                            label="Status", lines=2, interactive=False
                        )

                        gr.Markdown("### 📚 Sections")
                        sections_output = gr.Textbox(
                            label="Generated Sections", lines=10, interactive=False
                        )

                        create_sections_btn = gr.Button("📖 Create Sections")

                    with gr.Column(scale=1):
                        gr.Markdown("### 🔊 Audio Generation")

                        generate_btn = gr.Button(
                            "🎧 Generate Audiobook", variant="primary", size="lg"
                        )

                        audio_output = gr.Audio(
                            label="Generated Audiobook", type="filepath"
                        )

                        gr.Markdown(
                            """
                            # ## ⚙️ 它是如何运作的
                            1. Upload a PDF, DOCX, or TXT file then click on upload button
                            2. Click "Create Sections" to break content into parts
                            3. Click "Generate Audiobook" to create the course audio
                            4. Switch to Chat tab to ask questions
                            """
                        )

                # 事件处理程序
                upload_btn.click(
                    fn=app.process_document,
                    inputs=[file_input],
                    outputs=[upload_status, sections_output],
                )

                create_sections_btn.click(
                    fn=app.create_sections, outputs=[upload_status, sections_output]
                )

                generate_btn.click(fn=app.generate_audiobook, outputs=[audio_output])

            # =================================================================
            # TAB 2：互动聊天
            # =================================================================
            with gr.TabItem("💬 Interactive Chat"):
                gr.Markdown(
                    """
                    # ## 询问有关课程材料的问题
                    After generating your audio, switch to chat mode to ask questions!
                    """
                )

                switch_mode_btn = gr.Button(
                    "🔄 Switch to Chat Mode", variant="secondary"
                )
                mode_status = gr.Textbox(label="Mode Status", interactive=False)

                with gr.Row():
                    with gr.Column(scale=3):
                        chatbot = gr.Chatbot(
                            label="Course Assistant", height=500, show_copy_button=True
                        )

                        with gr.Row():
                            msg_input = gr.Textbox(
                                label="Your Question",
                                placeholder="Ask a question about the course material...",
                                scale=4,
                            )
                            submit_btn = gr.Button("Send", variant="primary", scale=1)

                    with gr.Column(scale=1):
                        gr.Markdown("### 🎤 Voice Input")

                        audio_input = gr.Audio(
                            sources=["microphone"],
                            type="filepath",
                            label="Record Question",
                        )

                        voice_btn = gr.Button("🎤 Ask by Voice")

                        gr.Markdown("### 🛠️ Options")
                        clear_btn = gr.Button("🧹 Clear History")

                # 事件处理程序
                def user_message(message, history):
                    return "", history + [[message, None]]

                def bot_response(history):
                    if len(history) > 0:
                        message = history[-1][0]
                        response = app.chat_response(message, history[:-1])
                        history[-1][1] = response
                    return history

                switch_mode_btn.click(fn=app.switch_to_chat, outputs=[mode_status])

                msg_input.submit(
                    fn=user_message,
                    inputs=[msg_input, chatbot],
                    outputs=[msg_input, chatbot],
                ).then(fn=bot_response, inputs=[chatbot], outputs=[chatbot])

                submit_btn.click(
                    fn=user_message,
                    inputs=[msg_input, chatbot],
                    outputs=[msg_input, chatbot],
                ).then(fn=bot_response, inputs=[chatbot], outputs=[chatbot])

                def voice_message(audio, history):
                    if audio:
                        response = app.voice_chat_response(audio, history)
                        return history + [["🎤 (voice question)", response]]
                    return history

                voice_btn.click(
                    fn=voice_message, inputs=[audio_input, chatbot], outputs=[chatbot]
                )

                clear_btn.click(
                    fn=lambda: app.chat.clear_history() or [], outputs=[chatbot]
                )

            # =================================================================
            # 选项卡 3：系统信息
            # =================================================================
            with gr.TabItem("ℹ️ System Info"):
                gr.Markdown("### 📊 System Information")

                def get_system_info():
                    info = []
                    info.append(
                        f"**Device:** {'CUDA' if torch.cuda.is_available() else 'CPU'}"
                    )
                    if torch.cuda.is_available():
                        info.append(f"**GPU:** {torch.cuda.get_device_name(0)}")
                        mem = app.manager.get_memory_usage()
                        info.append(f"**VRAM Used:** {mem['allocated']} GB")
                        info.append(f"**VRAM Free:** {mem['free']} GB")

                    info.append("\n### Model Status")
                    info.append(
                        f"- LLM: {'✅ Loaded' if app.manager.llm_loaded else '❌ Not loaded'}"
                    )
                    info.append(
                        f"- TTS: {'✅ Loaded' if app.manager.tts_loaded else '❌ Not loaded'}"
                    )
                    info.append(
                        f"- ASR: {'✅ Loaded' if app.manager.asr_loaded else '❌ Not loaded'}"
                    )

                    return "\n".join(info)

                system_info_output = gr.Markdown(get_system_info())
                refresh_btn = gr.Button("🔄 Refresh")
                refresh_btn.click(fn=get_system_info, outputs=[system_info_output])

                gr.Markdown(
                    """
                    # ## 📋 型号信息

                    | Model | Type | Purpose |
                    |-------|------|---------|
                    | Qwen2.5-1.5B-Instruct | LLM | Text generation, Q&A |
                    | Microsoft speecht5_tts | Audio | Voice narration |
                    | Whisper Tiny | ASR | Voice input |
                    """
                )

    return demo

# 主入口点

# ===================================================================

In [ ]:
print("=" * 60)
print("🎓 Course Audio Generator - Google Colab Edition")
print("=" * 60)

demo = create_ui()

demo.launch(
    debug=True,
    share=True,
    show_error=True,
)